In [1]:
%pip install xgboost

   ---------------------------------------- 0.0/124.9 MB ? eta -:--:--
   -- ------------------------------------- 7.6/124.9 MB 46.8 MB/s eta 0:00:03
   ------ --------------------------------- 20.7/124.9 MB 56.8 MB/s eta 0:00:02
   --------- ------------------------------ 30.1/124.9 MB 53.1 MB/s eta 0:00:02
   ------------ --------------------------- 40.1/124.9 MB 51.0 MB/s eta 0:00:02
   --------------- ------------------------ 49.5/124.9 MB 50.0 MB/s eta 0:00:02
   ------------------ --------------------- 59.2/124.9 MB 49.6 MB/s eta 0:00:02
   ---------------------- ----------------- 69.2/124.9 MB 49.6 MB/s eta 0:00:02
   ------------------------- -------------- 79.7/124.9 MB 49.3 MB/s eta 0:00:01
   ---------------------------- ----------- 89.9/124.9 MB 49.5 MB/s eta 0:00:01
   ------------------------------- -------- 99.9/124.9 MB 49.4 MB/s eta 0:00:01
   ---------------------------------- ---- 110.1/124.9 MB 49.5 MB/s eta 0:00:01
   ------------------------------------- - 121.4/1

In [6]:
import pandas as pd
import numpy as np
import glob
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# XGBOOST Training

In [9]:
import pandas as pd
import numpy as np
import glob
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# ---------------------------------------------------------
# 1. LOAD TRAIN + TEST DATA FROM ALL 7 SITES
# ---------------------------------------------------------

train_files = sorted(glob.glob("./train_sets/site_*_train.csv"))
test_files  = sorted(glob.glob("./test_sets/site_*_test.csv"))

df_train = pd.concat([pd.read_csv(f) for f in train_files], ignore_index=True)
df_test  = pd.concat([pd.read_csv(f) for f in test_files], ignore_index=True)

print(f"Training samples: {len(df_train)}")
print(f"Testing samples : {len(df_test)}")

# ---------------------------------------------------------
# 2. FEATURES AND TARGETS
# ---------------------------------------------------------

features = [
    "O3_forecast", "NO2_forecast", "T_forecast", "q_forecast",
    "u_forecast", "v_forecast", "w_forecast",
    "NO2_satellite", "HCHO_satellite", "ratio_satellite",
    "year", "month", "day", "hour"
]

targets = ["O3_target", "NO2_target"]

X_train = df_train[features]
X_test  = df_test[features]

y_train = df_train[targets]
y_test = df_test[targets]


# ---------------------------------------------------------
# 3. TRAIN 2 SEPARATE XGBOOST MODELS (BEST PRACTICE)
# ---------------------------------------------------------

models = {}

params = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "max_depth": 9,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.5,
    "reg_alpha": 0.5,
    "objective": "reg:squarederror",
    "tree_method": "hist"   # fast CPU training
}

for target in targets:
    print(f"\n🔵 Training XGBoost model for: {target}")
    
    model = XGBRegressor(**params)
    model.fit(X_train, y_train[target])

    models[target] = model

    # Save model
    joblib.dump(model, f"xgb_{target}.pkl")
    print(f"✔ Saved model: xgb_{target}.pkl")


# ---------------------------------------------------------
# 4. PREDICT USING BOTH MODELS
# ---------------------------------------------------------

y_pred = np.column_stack([
    models["O3_target"].predict(X_test),
    models["NO2_target"].predict(X_test)
])

# ---------------------------------------------------------
# 5. EVALUATE PERFORMANCE
# ---------------------------------------------------------

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("\n=============================")
print("     XGBOOST PERFORMANCE     ")
print("=============================")
print(f"RMSE: {rmse:.3f}")
print(f"R²  : {r2:.3f}")

Training samples: 120172
Testing samples : 25752

🔵 Training XGBoost model for: O3_target
✔ Saved model: xgb_O3_target.pkl

🔵 Training XGBoost model for: NO2_target
✔ Saved model: xgb_NO2_target.pkl

     XGBOOST PERFORMANCE     
RMSE: 26.264
R²  : -0.449


# Added Lag features


In [10]:


print("Loading all 7 site files...")

# ----------------------------------------
# 1. LOAD ALL SITE DATA
# ----------------------------------------

files = sorted(glob.glob("./train_sets/site_*_train.csv"))

df_list = [pd.read_csv(f) for f in files]
df = pd.concat(df_list, ignore_index=True)

print("Total rows loaded:", len(df))


# ----------------------------------------
# 2. SORT BY TIME
# ----------------------------------------

df = df.sort_values(["year", "month", "day", "hour"]).reset_index(drop=True)


# ----------------------------------------
# 3. CREATE TIME FEATURES
# ----------------------------------------

df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

df["dayofyear"] = df["datetime"].dt.dayofyear
df["weekday"] = df["datetime"].dt.weekday
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)
df["season"] = (df["month"] % 12) // 3   # 0 winter, 1 spring, 2 summer, 3 autumn


# ----------------------------------------
# 4. ADD LAG FEATURES (CRITICAL)
# ----------------------------------------

lags = [1, 2, 3, 6, 12, 24]

for lag in lags:
    df[f"O3_lag_{lag}"] = df["O3_target"].shift(lag)
    df[f"NO2_lag_{lag}"] = df["NO2_target"].shift(lag)

# Drop rows with NaN lags
df = df.dropna().reset_index(drop=True)

print("Rows after adding lags:", len(df))


# ----------------------------------------
# 5. TRAIN/TEST SPLIT (Time-based)
# ----------------------------------------

train_ratio = 0.80
train_end = int(len(df) * train_ratio)

train_df = df.iloc[:train_end]
test_df  = df.iloc[train_end:]

print("Training rows:", len(train_df))
print("Testing rows :", len(test_df))


# ----------------------------------------
# 6. FEATURES AND TARGETS
# ----------------------------------------

features = [
    "O3_forecast", "NO2_forecast", "T_forecast", "q_forecast",
    "u_forecast", "v_forecast", "w_forecast",
    "NO2_satellite", "HCHO_satellite", "ratio_satellite",
    "year", "month", "day", "hour",
    "dayofyear", "weekday", "is_weekend", "season"
]

# Add lag feature names
features += [f"O3_lag_{l}" for l in lags]
features += [f"NO2_lag_{l}" for l in lags]

targets = ["O3_target", "NO2_target"]

X_train = train_df[features]
X_test  = test_df[features]

y_train = train_df[targets]
y_test = test_df[targets]


# ----------------------------------------
# 7. TRAIN XGBOOST MODELS (SEPARATE MODELS)
# ----------------------------------------

models = {}

params = {
    "n_estimators": 700,
    "learning_rate": 0.05,
    "max_depth": 9,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_lambda": 1.5,
    "reg_alpha": 0.5,
    "objective": "reg:squarederror",
    "tree_method": "hist"
}

for target in targets:
    print(f"\n🔵 Training XGBoost for {target} ...")

    model = XGBRegressor(**params)
    model.fit(X_train, y_train[target])

    models[target] = model
    joblib.dump(model, f"xgb_{target}_with_lag.pkl")
    print(f"✔ Saved xgb_{target}_with_lag.pkl")


# ----------------------------------------
# 8. PREDICT AND EVALUATE
# ----------------------------------------

y_pred = np.column_stack([
    models["O3_target"].predict(X_test),
    models["NO2_target"].predict(X_test)
])

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print("\n=================================")
print("        FINAL XGBOOST SCORE      ")
print("=================================")
print(f"RMSE: {rmse:.3f}")
print(f"R²  : {r2:.3f}")
print("=================================")


Loading all 7 site files...
Total rows loaded: 120172
Rows after adding lags: 120148
Training rows: 96118
Testing rows : 24030

🔵 Training XGBoost for O3_target ...
✔ Saved xgb_O3_target_with_lag.pkl

🔵 Training XGBoost for NO2_target ...
✔ Saved xgb_NO2_target_with_lag.pkl

        FINAL XGBOOST SCORE      
RMSE: 19.178
R²  : 0.586


# XGBOOST Testing

# Added lag features for testing data

In [24]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, r2_score
import glob
import os

print("Loading test files...")

# ---------------------------------------------------------
# 1. LOAD ALL 7 TEST FILES
# ---------------------------------------------------------
test_files = sorted(glob.glob("./test_sets/site_*_test.csv"))
df_list = []

for f in test_files:
    print("Loaded:", f)
    df_list.append(pd.read_csv(f))

df = pd.concat(df_list, ignore_index=True)
print("Total test rows loaded:", len(df))


# ---------------------------------------------------------
# 2. SORT BY TIME
# ---------------------------------------------------------
df = df.sort_values(["year", "month", "day", "hour"]).reset_index(drop=True)


# ---------------------------------------------------------
# 3. CREATE TIME FEATURES (SAME AS TRAINING)
# ---------------------------------------------------------

df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

df["dayofyear"] = df["datetime"].dt.dayofyear
df["weekday"] = df["datetime"].dt.weekday
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)
df["season"] = (df["month"] % 12) // 3   # 0 winter, 1 spring, 2 summer, 3 autumn


# ---------------------------------------------------------
# 4. ADD LAG FEATURES (MUST MATCH TRAINING)
# ---------------------------------------------------------

lags = [1, 2, 3, 6, 12, 24]

for lag in lags:
    df[f"O3_lag_{lag}"]  = df["O3_target"].shift(lag)
    df[f"NO2_lag_{lag}"] = df["NO2_target"].shift(lag)

df = df.dropna().reset_index(drop=True)
print("Rows after adding lag features:", len(df))


# ---------------------------------------------------------
# 5. DEFINE FEATURES (EXACT TRAINING ORDER)
# ---------------------------------------------------------

features = [
    "O3_forecast", "NO2_forecast", "T_forecast", "q_forecast",
    "u_forecast", "v_forecast", "w_forecast",
    "NO2_satellite", "HCHO_satellite", "ratio_satellite",
    "year", "month", "day", "hour",
    "dayofyear", "weekday", "is_weekend", "season"
]

# add lag features
features += [f"O3_lag_{l}" for l in lags]
features += [f"NO2_lag_{l}" for l in lags]

X_test = df[features]
y_test_o3 = df["O3_target"]
y_test_no2 = df["NO2_target"]


# ---------------------------------------------------------
# 6. LOAD TRAINED MODELS
# ---------------------------------------------------------

model_o3  = joblib.load("xgb_O3_target_with_lag.pkl")
model_no2 = joblib.load("xgb_NO2_target_with_lag.pkl")

print("Models loaded successfully.")


# ---------------------------------------------------------
# 7. MAKE PREDICTIONS
# ---------------------------------------------------------

pred_o3  = model_o3.predict(X_test)
pred_no2 = model_no2.predict(X_test)


# ---------------------------------------------------------
# 8. EVALUATE
# ---------------------------------------------------------

def evaluate(true, pred, name):
    rmse = np.sqrt(mean_squared_error(true, pred))
    r2   = r2_score(true, pred)
    print("\n===============================")
    print(f"   XGBoost TEST SCORE: {name}")
    print("===============================")
    print(f"RMSE : {rmse:.3f}")
    print(f"R²   : {r2:.3f}")
    print("===============================\n")

evaluate(y_test_o3, pred_o3, "O3_target")
evaluate(y_test_no2, pred_no2, "NO2_target")


# ---------------------------------------------------------
# 9. SAVE PREDICTIONS
# ---------------------------------------------------------

df_out = df.copy()
df_out["O3_predicted"]  = pred_o3
df_out["NO2_predicted"] = pred_no2

df_out.to_csv("test_predictions_with_lags.csv", index=False)
print("✔ Saved test_predictions_with_lags.csv")

Loading test files...
Loaded: ./test_sets\site_1_test.csv
Loaded: ./test_sets\site_2_test.csv
Loaded: ./test_sets\site_3_test.csv
Loaded: ./test_sets\site_4_test.csv
Loaded: ./test_sets\site_5_test.csv
Loaded: ./test_sets\site_6_test.csv
Loaded: ./test_sets\site_7_test.csv
Total test rows loaded: 25752
Rows after adding lag features: 25728
Models loaded successfully.

   XGBoost TEST SCORE: O3_target
RMSE : 16.106
R²   : 0.476


   XGBoost TEST SCORE: NO2_target
RMSE : 17.969
R²   : 0.285

✔ Saved test_predictions_with_lags.csv
